# Parent Atlas — EmbeddingGemma ontology relation adapter (Colab)

## What are we training / making?
Freeze EmbeddingGemma and train only a small predicate-specific translation vector. For a judged triple (subject, relation, object), score `cosine(normalize(subject_embedding + relation_offset), object_embedding)`. This produces an experimental relation scorer, **not** a new embedding model and **not OnT**: it has no hyperbolic geometry or OnT role-rotation model. It does not update the base encoder.

This notebook pins the first baseline `google/embeddinggemma-300m` to immutable Hub commit `57c266a740f537b4dc058e1b0cda161fd15afa75`. To test `google/embeddinggemma-2`, use a separate run and its own immutable commit SHA. Both may emit 768D vectors, but vectors are not interchangeable across models. Do not merge them or overwrite canonical `semantic_768`. Before running, accept Google's Hugging Face terms and authenticate the Colab session if requested; never paste an access token into notebook output.

Input is a reviewed, approved JSONL export of positive and explicitly reviewed negative ontology triples, plus a SHA-256 sidecar bound to the workspace/taxonomy revisions. This notebook checks integrity and split hygiene, **not the export issuer's authority**. It enforces one split per source and leakage group. Do not treat missing ontology edges as negatives. The currently reported 4 parent edges are too few: this notebook requires >=20 training rows and >=10 each validation/test, both labels per split, and will stop otherwise.

Outputs: local Colab adapter NPZ + experiment receipt only. The receipt always says `canonical_authority=false`, `evidence_admitted=false`; no Parent Atlas store/index is written.

In [ ]:
MODEL_ID = "google/embeddinggemma-300m"
MODEL_REVISION = "57c266a740f537b4dc058e1b0cda161fd15afa75"
# For a challenger, set both MODEL_ID and MODEL_REVISION to an immutable Hub revision.
SEED, EPOCHS, LEARNING_RATE = 20261024, 80, 0.03
MIN_TRAIN, MIN_HELDOUT = 20, 10
%pip -q install -U "sentence-transformers>=5.1" scikit-learn

In [ ]:
import hashlib, json, random, zipfile, sentence_transformers
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics import average_precision_score, roc_auc_score
from google.colab import files
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print({"model": MODEL_ID, "device": DEVICE, "cuda": torch.cuda.is_available()})

## Dataset format
Upload one `.jsonl` and one `.receipt.json`. Every JSONL row must contain: `subject_id`, `subject_text`, `predicate_id`, `predicate_text`, `object_id`, `object_text`, integer `label` (0/1), `judgment` (`REVIEWED_NEGATIVE`/`REVIEWED_POSITIVE`), `source_ref`, `source_sha256`, `span_start_byte`, `span_end_byte`, `evidence_text`, `source_revision`, `workspace_revision`, `taxonomy_revision`, `evidence_ref`, `admission_receipt_id`, `split` (`train`/`validation`/`test`), and `leakage_group_id`. One frozen workspace and taxonomy revision are required. The sidecar must contain the exact `dataset_sha256` and an `export_receipt_id`. The notebook checks the excerpt's UTF-8 byte length against the span width, not the source file bytes or issuer authority; presence and hash do not independently prove admission.

In [ ]:
uploaded = files.upload()
jsonl = [n for n in uploaded if n.endswith(".jsonl")]
sidecars = [n for n in uploaded if n.endswith(".receipt.json")]
if len(jsonl) != 1 or len(sidecars) != 1: raise ValueError("Upload exactly one JSONL and one .receipt.json")
raw = Path(jsonl[0]).read_bytes(); digest = hashlib.sha256(raw).hexdigest()
export_receipt = json.loads(Path(sidecars[0]).read_text(encoding="utf-8"))
if export_receipt.get("dataset_sha256") != digest: raise ValueError("Dataset hash mismatch")
rows = [json.loads(line) for line in raw.decode("utf-8").splitlines() if line.strip()]
required = {"subject_id","subject_text","predicate_id","predicate_text","object_id","object_text","label","judgment","source_ref","source_sha256","span_start_byte","span_end_byte","evidence_text","source_revision","workspace_revision","taxonomy_revision","evidence_ref","admission_receipt_id","split","leakage_group_id"}
if not rows or any(not required.issubset(r) for r in rows): raise ValueError("Empty data or required fields missing")
for r in rows:
    if type(r["label"]) is not int or r["label"] not in (0,1): raise ValueError("label must be integer 0 or 1")
    expected = "REVIEWED_POSITIVE" if r["label"] else "REVIEWED_NEGATIVE"
    if r["judgment"] != expected: raise ValueError("Unreviewed or inconsistent label")
    if r["split"] not in {"train","validation","test"}: raise ValueError("Invalid split")
    if any(not str(r[k]).strip() for k in required - {"label"}): raise ValueError("Empty required field")
    if type(r["span_start_byte"]) is not int or type(r["span_end_byte"]) is not int or r["span_start_byte"] < 0 or r["span_end_byte"] <= r["span_start_byte"]: raise ValueError("Invalid evidence byte span")
    if len(r["evidence_text"].encode("utf-8")) != r["span_end_byte"]-r["span_start_byte"]: raise ValueError("Evidence excerpt byte length does not match span width")
    if len(r["source_sha256"]) != 64 or any(ch not in "0123456789abcdefABCDEF" for ch in r["source_sha256"]): raise ValueError("source_sha256 must be 64 hex characters")
if len({r["workspace_revision"] for r in rows}) != 1 or len({r["taxonomy_revision"] for r in rows}) != 1: raise ValueError("Dataset must use one workspace and taxonomy revision")
if not isinstance(export_receipt.get("export_receipt_id"), str) or not export_receipt["export_receipt_id"].strip(): raise ValueError("Missing export receipt identity")
if export_receipt.get("workspace_revision") != rows[0]["workspace_revision"] or export_receipt.get("taxonomy_revision") != rows[0]["taxonomy_revision"]: raise ValueError("Receipt revision binding mismatch")
groups, triples, source_bindings = {}, {}, {}
for r in rows:
    group = groups.setdefault(r["leakage_group_id"], r["split"])
    if group != r["split"]: raise ValueError("Leakage group crosses splits")
    source_binding = (r["source_revision"], r["source_sha256"].lower(), r["split"])
    previous_binding = source_bindings.setdefault(r["source_ref"], source_binding)
    if previous_binding != source_binding: raise ValueError("Source revision/hash or source-level split mismatch")
    key = (r["subject_id"], r["predicate_id"], r["object_id"])
    if key in triples: raise ValueError("Duplicate triple; deduplicate upstream and preserve its evidence lineage")
    triples[key] = r["split"]
splits = {s: [r for r in rows if r["split"] == s] for s in ("train","validation","test")}
for name, part in splits.items():
    if {r["label"] for r in part} != {0,1}: raise ValueError(f"{name} needs positive and negative judgments")
train_predicates = {r["predicate_id"] for r in splits["train"]}
if any(r["predicate_id"] not in train_predicates for s in ("validation","test") for r in splits[s]): raise ValueError("Every evaluated predicate must have training examples")
if len(splits["train"]) < MIN_TRAIN or any(len(splits[s]) < MIN_HELDOUT for s in ("validation","test")): raise ValueError("Insufficient rows for training/evaluation; no model was trained")
print({s: {"n":len(p), "positive":sum(r["label"]==1 for r in p), "negative":sum(r["label"]==0 for r in p)} for s,p in splits.items()})

In [ ]:
if len(MODEL_REVISION) != 40 or any(ch not in "0123456789abcdef" for ch in MODEL_REVISION): raise ValueError("MODEL_REVISION must be an immutable 40-character Hub commit SHA")
model = SentenceTransformer(MODEL_ID, revision=MODEL_REVISION, device=DEVICE)
model.eval()
transformer = model[0]
model_revision = getattr(getattr(getattr(transformer, "auto_model", None), "config", None), "_commit_hash", None)
if model_revision != MODEL_REVISION: raise ValueError("Loaded model revision does not match requested immutable revision")
entity_text, predicate_text = {}, {}
for r in rows:
    for table, key, value in ((entity_text,r["subject_id"],r["subject_text"]),(entity_text,r["object_id"],r["object_text"]),(predicate_text,r["predicate_id"],r["predicate_text"])):
        if key in table and table[key] != value: raise ValueError(f"Conflicting text for ID {key}")
        table[key] = value
entity_ids, predicate_ids = list(entity_text), list(predicate_text)
texts = list(entity_text.values()) + list(predicate_text.values())
inputs = [f"task: sentence similarity | query: {t}" for t in texts]
vectors = model.encode(inputs, convert_to_numpy=True, normalize_embeddings=True, batch_size=32, show_progress_bar=True)
if vectors.ndim != 2 or vectors.shape[1] != 768 or not np.isfinite(vectors).all(): raise ValueError(f"Expected finite 768D embeddings, got {vectors.shape}")
ei = {x:i for i,x in enumerate(entity_ids)}; pi = {x:i for i,x in enumerate(predicate_ids)}
ev = vectors[:len(entity_ids)]; pv = vectors[len(entity_ids):]
def tensors(part):
    s=np.stack([ev[ei[r["subject_id"]]] for r in part]); p=np.array([pi[r["predicate_id"]] for r in part]); o=np.stack([ev[ei[r["object_id"]]] for r in part]); y=np.array([r["label"] for r in part],dtype=np.float32)
    return torch.tensor(s,dtype=torch.float32,device=DEVICE),torch.tensor(p,dtype=torch.long,device=DEVICE),torch.tensor(o,dtype=torch.float32,device=DEVICE),torch.tensor(y,dtype=torch.float32,device=DEVICE)
train, validation, test = (tensors(splits[n]) for n in ("train","validation","test"))

## Train and compare
The baseline is raw EmbeddingGemma cosine similarity between subject and object text. The challenger adds a learned predicate offset. Report validation and held-out test AUROC/Average Precision; do not promote based on training loss or a single split.

In [ ]:
class RelationAdapter(torch.nn.Module):
    def __init__(self):
        super().__init__(); self.offsets=torch.nn.Parameter(torch.tensor(pv,dtype=torch.float32,device=DEVICE)*0.01); self.log_scale=torch.nn.Parameter(torch.tensor(2.0,device=DEVICE))
    def forward(self,s,p,o):
        moved=torch.nn.functional.normalize(s+self.offsets[p],dim=-1); return (moved*o).sum(-1)*self.log_scale.exp().clamp(max=100.0)
adapter=RelationAdapter().to(DEVICE); opt=torch.optim.AdamW(adapter.parameters(),lr=LEARNING_RATE,weight_decay=1e-3); loss_fn=torch.nn.BCEWithLogitsLoss()
def evaluate(batch):
    adapter.eval()
    with torch.no_grad(): logits=adapter(*batch[:3]).cpu().numpy()
    labels=batch[3].cpu().numpy()
    return {"auroc":float(roc_auc_score(labels,logits)),"average_precision":float(average_precision_score(labels,logits))},logits
for epoch in range(EPOCHS):
    adapter.train(); opt.zero_grad(set_to_none=True); loss=loss_fn(adapter(*train[:3]),train[3]); loss.backward(); opt.step()
    if epoch % 10 == 0 or epoch == EPOCHS-1: print({"epoch":epoch+1,"train_bce":round(float(loss.detach().cpu()),5),"validation":evaluate(validation)[0]})
challenger_metrics, challenger_logits=evaluate(test)
baseline_scores=(test[0]*test[2]).sum(-1).cpu().numpy(); labels=test[3].cpu().numpy()
baseline_metrics={"auroc":float(roc_auc_score(labels,baseline_scores)),"average_precision":float(average_precision_score(labels,baseline_scores))}
metrics={"baseline_frozen_text_cosine":baseline_metrics,"relation_adapter":challenger_metrics,"test_rows":len(labels)}
print(json.dumps(metrics,indent=2))

## Export experiment artifact
The export is for review/replay only. The model artifact ID, data hash, adapter hash, split counts, and metrics are recorded; authority remains explicitly false.

In [ ]:
out=Path("/content/parent-atlas-ontology-adapter"); out.mkdir(parents=True,exist_ok=True)
adapter_file=out/"relation-adapter.npz"
np.savez_compressed(adapter_file,offsets=adapter.offsets.detach().cpu().numpy(),predicate_ids=np.asarray(predicate_ids),log_scale=np.asarray(float(adapter.log_scale.detach().cpu())))
adapter_sha=hashlib.sha256(adapter_file.read_bytes()).hexdigest()
receipt={"schema":"parent-atlas.embeddinggemma-ontology-adapter-experiment.v1","created_at":datetime.now(timezone.utc).isoformat(),"model_id":MODEL_ID,"model_revision":model_revision,"sentence_transformers_version":sentence_transformers.__version__,"torch_version":torch.__version__,"representation_id":f"experimental:{MODEL_ID}:sentence-similarity-relation-translation:v1","embedding_prompt_policy":"task: sentence similarity | query: {text}","dimension":int(vectors.shape[1]),"seed":SEED,"epochs":EPOCHS,"learning_rate":LEARNING_RATE,"dataset_sha256":digest,"export_receipt_id_claim":export_receipt.get("export_receipt_id"),"issuer_independently_authenticated":False,"split_counts":{k:len(v) for k,v in splits.items()},"metrics":metrics,"adapter_sha256":adapter_sha,"canonical_authority":False,"evidence_admitted":False,"production_index_write":False}
receipt_file=out/"experiment-receipt.json"; receipt_file.write_text(json.dumps(receipt,indent=2,sort_keys=True)+"\n",encoding="utf-8")
zip_path=Path("/content/parent-atlas-ontology-adapter.zip")
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    z.write(adapter_file,adapter_file.name); z.write(receipt_file,receipt_file.name)
print(json.dumps(receipt,indent=2)); files.download(str(zip_path))